<a href="https://colab.research.google.com/github/simjonghyeon04/-/blob/main/4_10%20%EA%B3%BC%EC%A0%9C%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# 1. 패키지 및 크롬 브라우저 설치 (기존 콘솔 로그가 다시 길게 나오지 않도록 -q 옵션 유지)
!pip install -q selenium webdriver_manager
!apt-get update -qq
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add - > /dev/null 2>&1
!echo "deb http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update -qq
!apt-get install -y -qq google-chrome-stable

# 2. 크롬 옵션 설정 (안정적인 크롤링을 위해 유저 에이전트 추가)
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

try:
    url = "https://comic.naver.com/webtoon"
    driver.get(url)

    wait = WebDriverWait(driver, 10)
    # 실시간 신규 웹툰 영역을 정확히 포커싱
    ul_selector = "#container > div.ListSpot__spot_wrap--Iko15 > div.content > div > ul"
    target_ul = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ul_selector)))

    items = target_ul.find_elements(By.TAG_NAME, "li")

    print("=== 실시간 신규 웹툰 3개 리포트 ===\n")

    for i, item in enumerate(items[:3], 1):
        # 1. 불필요한 공백을 제거하고 유효한 텍스트 줄만 필터링
        lines = [line.strip() for line in item.text.split("\n") if line.strip()]

        # 2. 수집을 방해하는 배지 단어(신작, UP, 휴재 등)들을 리스트에서 완전히 제외
        ignore_keywords = ["신작", "UP", "휴재", "동결", "성인"]
        lines = [line for line in lines if line not in ignore_keywords]

        # 3. 텍스트가 유실되지 않도록 안정적인 인덱싱 매핑
        if len(lines) >= 3:
            title = lines[0]
            author = lines[1]
            # 만약 내용이 길어서 여러 줄로 쪼개졌다면 공백으로 합쳐줍니다.
            description = " ".join(lines[2:])
        elif len(lines) == 2:
            title = lines[0]
            author = lines[1]
            description = "내용 요약이 없습니다."
        else:
            # 데이터 구조가 깨진 항목은 패스
            continue

        # 원하시는 출력 포맷과 완벽하게 일치
        print(f"[{i}번 웹툰]")
        print(f"제목 : {title}")
        print(f"작가 : {author}")
        print(f"내용 : {description}")
        print("-" * 40)

except Exception as e:
    print(f"크롤링 중 오류가 발생했습니다: {e}")

finally:
    time.sleep(2)
    driver.quit()

W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt/sources.list.d/google-chrome.list:2
W: Target Packages (main/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt/sources.list.d/google-chrome.list:2
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt/sources.list.d/google-chrome.list:2
W: Target Packages (main/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/g